## Agent with Tool 

In [ ]:
search_tool = TavilySearch(max_results=5,topic='general',search_depth='basic')


llm = ChatOpenAI(model="gpt-4o", temperature=0.0, openai_api_key=api_key)


chat_fda_agent = create_react_agent(
        model=llm,
        tools=[search_tool],
        prompt=(
            "You are an FDA and Medical Device Expert. Provide accurate, authoritative, "
            "and professional answers on medical devices. Do NOT produce any NSFW, "
            "explicit, or inappropriate content.\n"
            "Use the chat history to maintain context.\n\n"
            "For general questions beyond FDA or medical device topics, you should still answer, "
            "but also encourage the user to ask about FDA regulations or medical devices.\n\n"
            "If the question pertains directly to FDA or medical-device details (e.g. approvals, "
            "regulations, safety), you may use the web search_tool to fetch up‑to‑date information.\n\n"
            "Final Answer: Please respond concisely and factually."
        )
    )

def genericChat(user_input,chat_history):
    try:

        user_msg = {"role": "user", "content": f"Respond to this : {user_input} with chat history for context {chat_history}"}

        ai_content = ""
        llm_response = chat_fda_agent.invoke({"messages": [user_msg]})
        assistant_msg = llm_response["messages"][-1]
        if isinstance(assistant_msg, AIMessage):
            ai_content = assistant_msg.content

        return ai_content

    except Exception as e:
        print("\nReason_llm : Error\n")
        error = f"LLM reasoning failed: {str(e)}"
        return error

## SQL AGent

In [ ]:
def load_agent_executor():
    """
    Loads the SQL Agent Executor.
    An agent is more robust and flexible than a simple chain.
    """
    toolkit = SQLDatabaseToolkit(db=db, llm=sqlllm)
    agent = create_sql_agent(
        llm=sqlllm,
        toolkit=toolkit,
        verbose=True,
        agent_type="openai-tools",
        agent_executor_kwargs={"return_intermediate_steps": True},
    )
    return agent


agent_executor = load_agent_executor()

## Async PosgresDB Setup with FastAPI

In [ ]:
from contextlib import asynccontextmanager
from psycopg_pool import AsyncConnectionPool
from langchain_postgres import PostgresChatMessageHistory
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")
TABLE_NAME = os.getenv("TABLE_NAME")


@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.db_pool = AsyncConnectionPool(DATABASE_URL, open=False)
    await app.state.db_pool.open(wait=True, timeout=10)
    async with app.state.db_pool.connection() as conn:
        await PostgresChatMessageHistory.acreate_tables(conn, TABLE_NAME)
    yield
    await app.state.db_pool.close()



app = FastAPI(lifespan=lifespan)


# --- Chat endpoint ---
@app.post("/generic_chat", response_model=ChatResponse)
async def chat_endpoint(req: ChatRequest):
    try:

        if not req.session_id: 
            session_id = str(uuid.uuid4())
            async with app.state.db_pool.connection() as conn:

                history = PostgresChatMessageHistory(
                    TABLE_NAME,
                    session_id,
                    async_connection=conn
                )

            msgs = await history.aget_messages()

            print("Length of Chat History : ",len(msgs))

            aimessage = genericChat(user_input=req.user_input,chat_history=msgs)

            await history.aadd_messages([HumanMessage(content=req.user_input)])
            await history.aadd_messages([AIMessage(content=aimessage)])

            return ChatResponse(session_id = session_id,response = aimessage)

## FastAPI Endpoint Example

In [ ]:
class SQLQueryRequest(BaseModel):
    query: str

class SQLQueryResponse(BaseModel):
    user_query : str
    final_answer: str
    sql_query: str
    table_result: str

@app.post("/get_sql_query", response_model=SQLQueryResponse)
async def get_sql_query_endpoint(request: SQLQueryRequest):
    try:
        final_answer, sql_query, table_result = get_sql_query(request.query)
        
        if sql_query is None:
            raise HTTPException(status_code=500, detail="Failed to generate SQL query.")
        
        return {
            "user_query": request.query,
            "final_answer": final_answer,
            "sql_query": sql_query, 
            "table_result": table_result
            }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

## Re-Ranker with Hybrid Search and Filtering

In [ ]:
from typing import List, Dict, Any, Optional
from collections import defaultdict
import numpy as np
from pydantic import Field, PrivateAttr
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize
from langchain.schema import BaseRetriever, Document
import nltk

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

class EnhancedGDNCRetriever(BaseRetriever):
    """Enhanced retriever with hybrid search (BM25 + semantic) and source-aware reranking.
    
    Combines the strengths of sparse (BM25) and dense (semantic) retrieval,
    with configurable weights and source-aware filtering.
    """
    
    # Required parameters
    base_retriever: BaseRetriever = Field(..., description="Base retriever (e.g., FAISS)")
    
    # Optional parameters with defaults
    k: int = Field(default=5, description="Number of documents to return")
    rerank: bool = Field(default=True, description="Enable/disable reranking")
    model_name: str = Field(
        default="cross-encoder/ms-marco-MiniLM-L-12-v2",
        description="Cross-encoder model for reranking"
    )
    bm25_weight: float = Field(
        default=0.3,
        ge=0.0,
        le=1.0,
        description="Weight for BM25 score in hybrid search (will be normalized with semantic_weight)"
    )
    min_score: float = Field(
        default=0.1,
        ge=0.0,
        le=1.0,
        description="Minimum score threshold for including results"
    )
    
    # Private attributes (not part of the model schema)
    _reranker: Any = PrivateAttr(default=None)
    _bm25: Any = PrivateAttr(default=None)
    
    def __init__(self, **data):
        super().__init__(**data)
        # Initialize the reranker model
        self._reranker = CrossEncoder(self.model_name)
        

    def _should_skip_doc(self, doc: Document) -> bool:
        """Check if document should be skipped (index/reference/title pages)"""
        if not doc.page_content.strip():
            return True
            
        # Get source and page info from metadata
        source = str(doc.metadata.get('source', '')).lower()
        page_content = doc.page_content.lower()
        
        # Skip conditions
        skip_keywords = ['index', 'reference', 'contents', 'title page', 'table of contents']
        if any(keyword in source for keyword in skip_keywords):
            return True
            
        # Check page content for common indicators
        content_indicators = [
            'this page intentionally left blank',
            'table of contents',
            'index',
            'references',
            'title page'
        ]
        
        return any(indicator in page_content for indicator in content_indicators)

    def _compute_scores(self, query: str, docs: List[Document]) -> List[float]:
        """Compute combined BM25 and semantic scores for documents"""
        if not docs:
            return []
            
        # Tokenize for BM25
        tokenized_corpus = [word_tokenize(doc.page_content.lower()) for doc in docs]
        self._bm25 = BM25Okapi(tokenized_corpus)
        
        # Get BM25 scores
        tokenized_query = word_tokenize(query.lower())
        bm25_scores = self._bm25.get_scores(tokenized_query)
        
        # Get semantic scores
        pairs = [(query, doc.page_content) for doc in docs]
        semantic_scores = self._reranker.predict(pairs)
        
        # Normalize scores to [0, 1] range
        def normalize(scores):
            if not scores.size:
                return scores
            min_val, max_val = np.min(scores), np.max(scores)
            return (scores - min_val) / (max_val - min_val + 1e-9)
            
        bm25_norm = normalize(np.array(bm25_scores))
        semantic_norm = normalize(np.array(semantic_scores))
        
        # Combine scores using weighted sum
        combined = (self.bm25_weight * bm25_norm) + ((1 - self.bm25_weight) * semantic_norm)
        
        return combined.tolist()

    def get_relevant_documents(self, query: str) -> List[Document]:
        """Retrieve and rerank documents based on the query without modifying them"""
        # Get initial documents from base retriever
        docs = self.base_retriever.invoke(query)
        if not docs or not self.rerank:
            return docs[:self.k]
        
        # Filter out index/reference/title pages
        filtered_docs = [doc for doc in docs if not self._should_skip_doc(doc)]
        if not filtered_docs:
            return []
        
        # Compute scores for all documents
        scores = self._compute_scores(query, filtered_docs)
        
        # Get source distribution for normalization
        sources = [str(doc.metadata.get('source', 'general')).lower() for doc in filtered_docs]
        source_counts = {}
        for src in sources:
            source_counts[src] = source_counts.get(src, 0) + 1
        max_source_count = max(source_counts.values()) if source_counts else 1
        
        # Score and sort documents
        scored_docs = []
        for doc, score, source in zip(filtered_docs, scores, sources):
            # Apply source-based weighting
            source_weight = 0.5 + (0.5 * (source_counts[source] / max_source_count))
            final_score = score * source_weight
            
            if final_score >= self.min_score:
                scored_docs.append((final_score, doc))
        
        # Sort by score (descending) and return top-k
        scored_docs.sort(key=lambda x: x[0], reverse=True)
        return [doc for score, doc in scored_docs[:self.k]]

    def _get_relevant_documents(self, query: str) -> List[Document]:
        """Required by BaseRetriever - calls the main retrieval logic"""
        return self.get_relevant_documents(query)

    async def aget_relevant_documents(self, query: str) -> List[Document]:
        """Async version for compatibility"""
        return self.get_relevant_documents(query)

In [ ]:
retriever_gdnc = load_faiss_db("vector_gdnc")

retriever_gdnc = retriever_gdnc.as_retriever(search_type="mmr", search_kwargs={"k": 15,"score_threshold": 0.7,"lambda_mult": 0.7}) if retriever_gdnc else None

retriever_gdnc = EnhancedGDNCRetriever(
    base_retriever=retriever_gdnc,
    k=8,                          # Final number of docs
    rerank = True
)


## Ensemble of VectorDB

In [ ]:
from langchain.retrievers import EnsembleRetriever


retriever_map = {
    "cfr": retriever_cfr,
    "sng": retriever_std,
    "gdnc": retriever_gdnc,
}

def make_ensemble(selected_keys, weights=None):
    retrievers = [retriever_map[k] for k in selected_keys if retriever_map.get(k) is not None]
    if not retrievers:
        print("No valid retrievers selected or loaded.")
        return None # Or raise an error, depending on desired behavior
    # If weights not provided, default equal weighting
    if weights is None:
        weights = [1] * len(retrievers)
    return EnsembleRetriever(retrievers=retrievers, weights=weights)

## Stream Output FastAPI

In [ ]:
async def stream_events(graph, inputs, config, thread_id: str):
    """
    Streams events from the graph, yielding the thread_id first,
    then the content from the 'reason_llm' node in real-time chunks.
    """
    replyFlag = False
    # 1. Send the thread_id as the first event
    yield f"data: {json.dumps({'thread_id': thread_id})}\n\n"
    print(f"\n--- Starting Graph Stream for Thread ID: {thread_id} ---\n")

    try:
        # 2. Stream each event from the graph
        events = graph.astream_events(input=inputs, config=config, version="v2")
        async for event in events:
            if event["event"] == "on_chat_model_stream" and len(event["tags"]) > 1 and event["tags"][1] == "RegulatoryExpert":
                replyFlag = True
                chunk = event["data"]["chunk"].content
                #print(chunk, end="", flush=True)
                yield f"data: {json.dumps({'response': chunk})}\n\n"

        if replyFlag == False:
            current_state = await graph.aget_state(config)
            if current_state.values["classification"]["classification"] == "generic":
                yield f"data: {json.dumps({'response': current_state.values['classification']['reply']})}\n\n"


    except Exception as e:
        print(f"!!! EXCEPTION in stream_events: {e}")
    finally:
        print(f"\n--- Finished Graph Stream for Thread ID: {thread_id} ---\n")

@app.post("/interact")
async def interact(request: InteractionRequest):
    # Handle empty or no input
    if (not request.user_choices or request.user_choices is None or request.user_choices == {} or 
        not request.user_input or request.user_input is None or str(request.user_input).strip() == ""):
        async def stream_empty_input_message():
            yield f"data: {json.dumps({'response': 'No input provided. Please enter a choice or input text.'})}\n\n"
        return StreamingResponse(stream_empty_input_message(), media_type="text/event-stream")
    
    thread_id = request.thread_id or str(uuid.uuid4())
    thread_config: RunnableConfig = {"configurable": {"thread_id": thread_id}}

    if not request.thread_id:
        # Start a new conversation
        state_input = initial_state.copy()
        state_input.update({
            'user_choices': request.user_choices,
            'feedback': request.user_input,
            'useDeviceData': request.useDeviceData,
            'userProvidedDeiveceData': request.userProvidedDeiveceData,
            'chat_history':  [RemoveMessage(id=REMOVE_ALL_MESSAGES)]
        })
        return StreamingResponse(stream_events(graph, state_input, thread_config, thread_id), media_type="text/event-stream")
   

## Test Streaming Output

In [ ]:
import requests
import json


url = "http://localhost:8025/interact"


data = {
    "user_choices" : user_choices,
    'thread_id': thread_id, # Uncomment to continue a conversation
    #"user_input": "Do not Deviate from the original path. At the last node, change it to 'N' and tell me the consequeces.",

    #"user_input": "the path you suggested from to 3 to 6 does not exist",
    "user_input": "If I had to start from first node what would be the best path ?",
    #"user_input" : "Hello, How are you ?",
    #"user_input": "I want to exit the conversation",
}


# Use stream=True to handle the streaming response
response = requests.post(url, json=data, stream=True)

# Check if the request was successful
if response.status_code == 200:
    thread_id_received = False
    for chunk in response.iter_lines():
        if chunk:
            # The chunk is in bytes, decode it
            chunk_str = chunk.decode('utf-8')
            if chunk_str.startswith('data:'):
                # Extract the JSON part
                json_data = chunk_str[len('data:'):].strip()
                if not json_data:
                    continue
                try:
                    data_dict = json.loads(json_data)
                    # The first message should be the thread_id
                    if not thread_id_received and 'thread_id' in data_dict:
                        print(f"Thread ID: {data_dict['thread_id']}\n---")
                        thread_id_received = True
                    # Subsequent messages are response tokens
                    if 'response' in data_dict:
                        print(data_dict.get('response', ''), end='', flush=True)
                except json.JSONDecodeError:
                    pass  # Ignore chunks that are not valid JSON
    print()  # Print a final newline
else:
    print(f"Error: {response.status_code}")
    print(response.text)

## Manage Chat History Interactions

In [ ]:
from typing import Dict,Annotated,List
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

class AgentState(TypedDict):
    tree : List[Dict]
    user_choices : Dict
    current_path_str: str
    user_decisions_str: str
    context_docs_str: str
    classification : Dict
    chat_history:  Annotated[list[AnyMessage], add_messages]  # this one

## Interrupt in Langgraph and Resume via FastAPI

In [ ]:
from langgraph.types import interrupt

def process_feedback(state) -> str:
    
    print("Node : process_feedback")

    feedback = interrupt("Please provide detailed feedback or type 'exit' to end:").strip().lower()
    state["feedback"] = "Not Available"
    
    state["feedback"] = feedback
    print(f"\n🧑 User Feedback:\n {state['feedback']}")


    return state

In [ ]:
@app.post("/interact")
async def interact(request: InteractionRequest):
    
    # Continue existing conversation
    current_state = await graph.aget_state(thread_config)
    
    # Check if the thread exists and is complete
    if not current_state.next:
        async def stream_completion_message():
            yield f"data: {json.dumps({'thread_id': thread_id})}\n\n"
            yield f"data: {json.dumps({'response': 'Graph execution is complete for this session.'})}\n\n"
        return StreamingResponse(stream_completion_message(), media_type="text/event-stream")

    inputs = Command(
            resume=request.user_input, 
            update={
                "useDeviceData": request.useDeviceData,
                'userProvidedDeiveceData': request.userProvidedDeiveceData
            }
        )
    return StreamingResponse(stream_events(graph, inputs, thread_config, thread_id), media_type="text/event-stream")


## Save Graph Chechpoint in Async Postgres with Langgraph and FastAPI 

In [ ]:
def decide_start_node(state):
    if state.get('classification') and state.get('classification')['classification'] == "exit":
        return "end"
    elif state.get('classification') and state.get('classification')['classification'] == "generic":
        return "feedbackloop"
    elif state.get('useDeviceData') == True and state['userProvidedDeiveceData']: 
        return "device" 
    elif state.get('useDeviceData') == False or state['userProvidedDeiveceData'] == "": 
        return "knowledge" 
    
     
    
def create_graph(checkpointer):
    graph = StateGraph(state_schema = AgentState)

    graph.add_node("knowledge_base",build_decision_tree_prompt)
    graph.add_node("reason_llm", reason_llm)
    graph.add_node("process_feedback", process_feedback)
    graph.add_node("device_summary", deviceSummary)
    graph.add_node("user_initpath", extract_user_decision_and_path)
    graph.add_node("classify_node", classify_node)
    
    
    graph.set_entry_point("user_initpath")
    graph.add_edge("user_initpath", "classify_node")
    graph.add_conditional_edges(
        "classify_node",
        decide_start_node,
        {
            "feedbackloop" : "process_feedback",
            "device": "device_summary",
            "knowledge": "knowledge_base",
            "end": END
        }
    )
    graph.add_edge("device_summary", "knowledge_base")
    graph.add_edge("knowledge_base", "reason_llm")
    graph.add_edge("reason_llm", "process_feedback")
    graph.add_edge("process_feedback", "classify_node")

    return graph.compile(checkpointer=checkpointer)


In [ ]:
DB_URI = os.getenv("DB_URI")

graph = None

@asynccontextmanager
async def lifespan(app: FastAPI):

    connection_kwargs = { "autocommit": True, "prepare_threshold": None}
    
    pool = AsyncConnectionPool(
        conninfo=DB_URI,
        kwargs=connection_kwargs,
        open=False
    )
    await pool.open()

    app.state.checkpointer = AsyncPostgresSaver(pool)

    await app.state.checkpointer.setup()

    global graph
    
    graph = create_graph(checkpointer=app.state.checkpointer)
    
    with open("static/graph.png", "wb") as f:
        f.write(graph.get_graph().draw_mermaid_png())

    yield
    await pool.close()

app = FastAPI(lifespan=lifespan)

## Specific JSON Output from Agent

In [ ]:

class Classify(BaseModel):
    classification: Literal["generic", "fda", "exit"] = Field(
        description="Classify the topic type: 'generic' or 'fda' or 'exit'"
    )
    reply: str = Field(
        description="If 'generic', a natural language reply to the query. If 'fda', respond with 'FDA'. If 'exit', respond with 'exit'."
    )

parser = PydanticOutputParser(pydantic_object=Classify)

format_instructions = parser.get_format_instructions()

agent = create_react_agent(
    model=llm,
    tools=[ddg_search_tool],
    prompt=(
        "You are an expert FDA decision-tree assistant.\n\n"
        "You MUST call the DuckDuckGoSearch tool at least once before replying.\n\n"
        "Your output MUST conform exactly to this JSON schema (no extra fields):\n"
        f"{format_instructions}\n\n"
        "Add a key `classification` with value either \"generic\" or \"fda\" or \"exit\":\n"
        "- If the query is a greeting or unrelated to FDA topics, set `classification` to \"generic\".\n"
        "- If the query is related to FDA topics (drug, medical, device, risk, path, node, regulatory, decision tree) set `classification` to \"fda\".\n\n"
        "- If the query is to exit the conversation, set `classification` to \"exit\".\n\n"
        "Also add a key `reply`:\n"
        "- If `classification` is 'generic', `reply` must be a natural language response to the query.\n"
        "- If `classification` is 'fda', `reply` must be the string \"FDA\".\n\n"
        "- If `classification` is 'exit', `reply` must be the string \"exit\".\n\n"
        "IMPORTANT: The output must be *only* the JSON object—no extra text or reasoning.\n"
    )
)


def AgentClassifyNode(topic):

    max_retries = 3
    attempt = 0

    user_msg = {"role": "user", "content": f"Classify this topic : {topic}"}

    while attempt < max_retries:
        
        attempt += 1
        
        
        # Run agent and capture full assistant output (stream or no-stream)
        llm_response = agent.invoke({"messages": [user_msg]})
        assistant_msg = llm_response["messages"][-1]
        ai_content = assistant_msg.content

        try :
            # Parse the final JSON into Pydantic model
            article: Classify = parser.parse(ai_content)
            return article.model_dump()

        except ValidationError as e:
            print(f"[Attempt {attempt}] Parsing failed:", e)
            # Optionally modify the prompt to highlight the error:
            user_msg += "\n\nNote: Your previous output did not match the required JSON schema. Please fix it exactly."
            continue

    # If all attempts fail, raise or return empty/default
    raise RuntimeError(f"Failed to get valid ArticleDraft JSON after {max_retries} attempts.")



## Delete Chat Sessions using FastAPI

In [ ]:
import os
from dotenv import load_dotenv

from sqlalchemy import MetaData, Table, delete, select
from sqlalchemy.orm import sessionmaker
from sqlalchemy import create_engine

load_dotenv()

DATABASE_URL = os.getenv("DB_URI")
TABLE_NAME = os.getenv("TABLE_NAME")


metadata = MetaData()

engine = create_engine(DATABASE_URL)
Session = sessionmaker(bind=engine)

session = Session()



# --- Delete chat session endpoint ---
@app.delete("/delete_session")
async def delete_session_endpoint(request: DeleteSession):
    try:
        table = Table(TABLE_NAME, metadata, autoload_with=engine)  # Validate table name
        
        # 1️⃣ Check if the session_id exists
        stmt_check = select(table).where(table.c.session_id == request.session_id)
        result = session.execute(stmt_check).first()

        if result is None:
            raise HTTPException(status_code=404, detail="No chat session found with this ID")

                
        stmt = delete(table).where(table.c.session_id == request.session_id)
        session.execute(stmt)
        session.commit()
        return {"message": "Chat session deleted successfully", "session_id": request.session_id}
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


## Delete graph Chechkpoint from PostgresDB using FastAPI

In [ ]:
@app.delete("/delete_thread")
async def delete_thread(request: DeleteThreadRequest):
    try:
        saver: AsyncPostgresSaver = app.state.checkpointer
        # first check if the thread exists using aget and RunnableConfig
        thread_config: RunnableConfig = {"configurable": {"thread_id": request.thread_id}}
        thread_exists = await saver.aget(thread_config)
        if not thread_exists:
            raise HTTPException(status_code=404, detail="Thread not found")
        await saver.adelete_thread(request.thread_id)
        return {"message": "Thread deleted successfully", "thread_id": request.thread_id}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e)) 
